In [ ]:
#| default_exp callbacks.core

# callbacks.core

> Platform infra + standard 11-rail CBs + topology guards.
> Auto-generated by nbdev from `nbs/api/callbacks/core.ipynb`.

In [ ]:
#| export
from __future__ import annotations
import copy
from fastcore.all import *
from operator import attrgetter
from cftime import date2num ,num2date
import numpy as np
import pandas as pd
from typing import List, Dict, Callable, Any, Optional, Union
from collections import defaultdict
from marisco.configs import get_lut, get_time_units, NC_GROUPS, SMP_TYPE_LUT
from pydantic import BaseModel, Field

## Foundation

In [ ]:
#| export
class Callback(): 
    "Base class for callbacks."
    order = 0
    def __init__(self): pass

In [ ]:
#| export
class PerGroupCB(Callback):
    "Calls `each_grp` for each group in `tfm.dfs`; set `grps` to restrict to specific groups."
    grps: list = None

    def __init__(self,
                 grps: list=None  # Groups to process; None = all groups in `tfm.dfs`
                 ):
        if grps is not None: self.grps = grps

    def __call__(self, tfm):
        for grp in (self.grps or tfm.dfs):
            if grp in tfm.dfs: self.each_grp(grp, tfm.dfs[grp], tfm)

In [ ]:
#| export
@patch
def each_grp(self:PerGroupCB,
             grp: str,          # Group key e.g. `'SEAWATER'`, `'BIOTA'`
             df: pd.DataFrame,  # DataFrame for this group
             tfm,               # Parent `Transformer`
             ):
    "Override to implement per-group transformation logic."
    raise NotImplementedError

In [ ]:
#| export
def run_cbs(
    cbs: List[Callback], # List of callbacks to run
    obj: Any # Object to pass to the callbacks
    ):
    "Run the callbacks in the order they are specified."
    for cb in sorted(cbs, key=attrgetter('order')):
        if cb.__doc__: obj.logs.append(cb.__doc__)
        cb(obj)

In [ ]:
#| export
class Transformer():
    "Transform the dataframe(s) according to the specified callbacks."
    def __init__(self, 
                 data: Union[Dict[str, pd.DataFrame], pd.DataFrame], # Data to be transformed
                 cbs: Optional[List[Callback]]=None, # List of callbacks to run
                 custom_maps: Dict = None,
                 inplace: bool=False # Whether to modify the dataframe(s) in place
                 ): 
        store_attr()
        self.is_single_df = isinstance(data, pd.DataFrame)
        self.df, self.dfs = self._prepare_data(data, inplace)
        self.logs = []
        self.custom_maps = custom_maps or defaultdict(lambda: defaultdict(dict))
            
    def _prepare_data(self, data, inplace):
        if self.is_single_df:
            return (data if inplace else data.copy()), None
        else:
            return None, (data if inplace else {k: v.copy() for k, v in data.items()})
    
    def unique(self, col_name: str) -> np.ndarray:
        "Distinct values of a specific column present in all groups."
        if self.is_single_df:
            values = self.df.get(col_name, pd.Series()).dropna().values  # grandfathered
        else:
            columns = [df.get(col_name) for df in self.dfs.values() if df.get(col_name) is not None]
            values = np.concatenate([col.dropna().values for col in columns]) if columns else []  # grandfathered
        return np.unique(values)
        
    def __call__(self):
        "Transform the dataframe(s) according to the specified callbacks."
        if self.cbs: run_cbs(self.cbs, self)
        return self.df if self.dfs is None else self.dfs

## Cleaning & validation

In [ ]:
#| export
class SanitizeLonLatCB(PerGroupCB):
    "Drop rows with invalid longitude & latitude values. Convert `,` separator to `.` separator."
    def __init__(self, 
                 lon_col: str='LON', # Longitude column name
                 lat_col: str='LAT', # Latitude column name
                 verbose: bool=False # Whether to print the number of invalid longitude & latitude values
                 ):
        store_attr()

    def each_grp(self, grp, df, tfm):
        df[self.lon_col] = df[self.lon_col].apply(lambda x: float(str(x).replace(',', '.')))
        df[self.lat_col] = df[self.lat_col].apply(lambda x: float(str(x).replace(',', '.')))
        mask_zeroes = (df[self.lon_col] == 0) & (df[self.lat_col] == 0)
        if mask_zeroes.sum() and self.verbose:
            print(f'The "{grp}" group contains {mask_zeroes.sum()} data points whose ({self.lon_col}, {self.lat_col}) = (0, 0)')
        mask_goob = (df[self.lon_col] < -180) | (df[self.lon_col] > 180) | (df[self.lat_col] < -90) | (df[self.lat_col] > 90)
        if mask_goob.sum() and self.verbose:
            print(f'The "{grp}" group contains {mask_goob.sum()} data points with unrealistic {self.lon_col} or {self.lat_col} values.')
        tfm.dfs[grp] = df.loc[~(mask_zeroes | mask_goob)]

## Value mapping

In [ ]:
#| export
class RemapCB(PerGroupCB):
    "Remap source values to MARIS standard identifiers using a lookup table."
    def __init__(self,
                 lut: dict|Callable,  # Lookup: dict, or callable(dfs)->dict
                 col_remap: str,            # Destination column to create
                 col_src: str,              # Source column with provider values
                 default_val: int=0,        # Value assigned to unmapped source values
                 grps: list[str]=None,      # Groups to process (None = all)
                ):
        store_attr()
        grp_str = ', '.join(str(g) for g in grps) if grps else 'all'
        self.__doc__ = f"Remap values from '{col_src}' to '{col_remap}' for groups: {grp_str}."

    def _resolve_lut(self, tfm):
        "Resolve the LUT: if a callable, call it with tfm's dfs to produce a dict."
        spec = self.lut
        if callable(spec):
            dfs = tfm.dfs if not tfm.is_single_df else {'_': tfm.df}
            spec = spec(dfs)
        return spec

    def __call__(self, tfm):
        self._resolved_lut = self._resolve_lut(tfm)
        super().__call__(tfm)

    def each_grp(self, grp, df, tfm):
        df[self.col_remap] = (df[self.col_src]
            .map(self._resolved_lut).fillna(self.default_val).astype(int))  # grandfathered

In [ ]:
#| export
class SoftRemapCB(PerGroupCB):
    "Remap source tokens to MARIS IDs; lut=None/empty maps all tokens to default_val (Null-Object no-op)."
    class Schema(BaseModel):
        col_src:    str
        col_remap:  str
        lut:        dict[str, Any] = Field(default_factory=dict)
        default_val: Any           = 0

    def __init__(self,
                 col_src:     str,
                 col_remap:   str,
                 lut:         dict=None,
                 default_val: Any=0,
                 grps: list[str]=None,
                ):
        self.cfg  = self.Schema(col_src=col_src, col_remap=col_remap,
                                lut=lut or {}, default_val=default_val)
        self.grps = [None] if not self.cfg.lut else grps

    def each_grp(self, grp, df, tfm):
        df[self.cfg.col_remap] = (df[self.cfg.col_src]
            .map(self.cfg.lut).fillna(self.cfg.default_val).astype(int))  # grandfathered

## Schema alignment

In [ ]:
#| export
class RenameColsCB(PerGroupCB):
    "Rename provider columns to MARIS standard names; optionally cast specified columns to str."
    class Schema(BaseModel):
        mapping:     dict[str, str] = Field(default_factory=dict)
        string_cast: list[str]      = Field(default_factory=list)
    def __init__(self, mapping: dict=None, string_cast: list=None):
        self.cfg = self.Schema(mapping=mapping or {}, string_cast=string_cast or [])
    def each_grp(self, grp, df, tfm):          # ZERO ast.If
        df.rename(columns=self.cfg.mapping, inplace=True)
        for col in self.cfg.string_cast:
            df[col] = df[col].astype(str)

## Wide-to-long reshaping

In [ ]:
#| export
class SoftMeltWideNuclidesCB(PerGroupCB):
    "Reshape wide nuclide columns to long format; spec=None/[] is a safe Null-Object no-op."
    class Schema(BaseModel):
        spec: list = Field(default_factory=list)

    def __init__(self, spec: list=None, grp: str='SEAWATER'):
        self.cfg  = self.Schema(spec=spec or [])
        self.grps = [grp]

    def each_grp(self, grp, df, tfm):
        frames = []
        for s in self.cfg.spec:
            sub = df.dropna(subset=[s['val']]).copy()  # grandfathered
            sub['NUCLIDE'] = s['nuclide']
            sub['VALUE']   = sub[s['val']]
            sub['UNC']     = sub[s['unc']]
            sub['UNIT']    = s['unit']
            sub['LAB']     = s['lab']
            frames.append(sub)
        tfm.dfs[grp] = pd.concat(frames or [df], ignore_index=True)

## Sample ID

In [ ]:
#| export
class AddSampleIDCB(PerGroupCB):
    "Assign 1-based sequential SMP_ID; optionally cast a provider ID column to str for NetCDF VLEN compatibility."
    def __init__(self,
                 col_provider: str=None,  # Provider ID column to cast to str; None = skip
                 ):
        store_attr()

    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.reset_index(drop=True)
        tfm.dfs[grp]['SMP_ID'] = tfm.dfs[grp].index + 1
        if self.col_provider:
            tfm.dfs[grp][self.col_provider] = tfm.dfs[grp][self.col_provider].astype(str).astype(object)

## Time

In [ ]:
#| export
class EncodeTimeCB(PerGroupCB):
    "Encode time as seconds since epoch."    
    def __init__(self, 
                   col_time: str='TIME',  # Time column name
                   verbose: bool=False,  # Print warning about missing time values
                   fn_units: Callable=get_time_units # Function returning the time units
                 ): 
        store_attr()
        self.units = fn_units()

    def each_grp(self, grp: str, df: pd.DataFrame, tfm):
        n_missing = df[self.col_time].isna().sum()
        if self.verbose and n_missing: print(f"Warning: {n_missing} missing time value(s) in {grp}")
        tfm.dfs[grp] = df[df[self.col_time].notna()]
        tfm.dfs[grp][self.col_time] = tfm.dfs[grp][self.col_time].apply(lambda x: date2num(x, units=self.units))

In [ ]:
#| export
class DecodeTimeCB(PerGroupCB):
    "Decode time from seconds since epoch to datetime format."    
    def __init__(self, 
                 col_time: str='TIME',
                 fn_units: Callable=get_time_units # Function returning the time units
                 ): 
        store_attr()
        self.units = fn_units()

    def each_grp(self, grp, df, tfm):
        n_missing = df[self.col_time].isna().sum()
        if n_missing: print(f"Warning: {n_missing} missing time value(s) in {grp}.")
        tfm.dfs[grp] = df[df[self.col_time].notna()]
        tfm.dfs[grp][self.col_time] = tfm.dfs[grp][self.col_time].apply(
            lambda x: num2date(x, units=self.units, only_use_cftime_datetimes=False)
        )

## Soft date-time & unit-convert

In [ ]:
#| export
class SoftParseDateTimeCB(PerGroupCB):
    "Parse date and time columns into a UTC-aware TIME column; col_date=None is a safe Null-Object no-op."
    class Schema(BaseModel):
        col_date: Optional[str] = None
        col_time: Optional[str] = None
        fmt: str = "%Y-%m-%d"
    def __init__(self, col_date=None, col_time=None, fmt="%Y-%m-%d"):
        self.cfg  = self.Schema(col_date=col_date, col_time=col_time, fmt=fmt)
        self.grps = None if col_date else [None]
    def each_grp(self, grp, df, tfm):          # ZERO ast.If
        date_str = df[self.cfg.col_date].astype(str)
        time_str = df.get(self.cfg.col_time, pd.Series("", index=df.index)).astype(str)
        df['TIME'] = pd.to_datetime((date_str + " " + time_str).str.strip(),
                                    format=self.cfg.fmt, utc=True)

In [ ]:
#| export
class SoftConvertUnitCB(PerGroupCB):
    "Apply a scalar unit conversion to rows matching a specific nuclide; rule=None is a safe Null-Object no-op."
    class UnitConversionRule(BaseModel):
        nuclide:  str
        src_unit: str
        dst_unit: str
        factor:   float
    def __init__(self, rule: dict = None):
        self.grps = [None] if rule is None else None
        if rule: self.cfg = self.UnitConversionRule(**rule)
    def each_grp(self, grp, df, tfm):          # ZERO ast.If
        mask = df['NUCLIDE'] == self.cfg.nuclide
        df.loc[mask, 'VALUE'] *= self.cfg.factor
        df.loc[mask, 'UNC']   *= self.cfg.factor
        df.loc[mask, 'UNIT']   = self.cfg.dst_unit

## Topology Guards (S-8b: promoted from general.py inner classes)

In [ ]:
#| export
class _GuardedEncodeTimeCB(EncodeTimeCB):
    "Topology-guarded EncodeTimeCB: auto-degrades to Null-Object (grps=[None]) when TIME absent."
    def __call__(self, tfm):
        self.grps = ([None] if any('TIME' not in df.columns for df in tfm.dfs.values())
                     else None)
        super().__call__(tfm)


class _GuardedSanitizeLonLatCB(SanitizeLonLatCB):
    "Topology-guarded SanitizeLonLatCB: auto-degrades to Null-Object when LON or LAT absent."
    def __call__(self, tfm):
        self.grps = ([None] if any('LON' not in df.columns or 'LAT' not in df.columns
                                   for df in tfm.dfs.values())
                     else None)
        super().__call__(tfm)

In [ ]:
# Topology Guard smoke-tests
import pandas as pd
from marisco.callbacks.core import (_GuardedEncodeTimeCB, _GuardedSanitizeLonLatCB,
                                     Transformer)

# _GuardedEncodeTimeCB: TIME absent -> Null-Object (grps=[None])
tfm = Transformer({'SEAWATER': pd.DataFrame({'LON': [1.0], 'LAT': [1.0]})}, cbs=[])
g = _GuardedEncodeTimeCB()
g(tfm)
assert g.grps == [None], f'Expected [None], got {g.grps}'
print('_GuardedEncodeTimeCB Null-Object guard: OK')

# _GuardedSanitizeLonLatCB: LON absent -> Null-Object
tfm2 = Transformer({'SEAWATER': pd.DataFrame({'TIME': [1000000]})}, cbs=[])
g2 = _GuardedSanitizeLonLatCB()
g2(tfm2)
assert g2.grps == [None], f'Expected [None], got {g2.grps}'
print('_GuardedSanitizeLonLatCB Null-Object guard: OK')